<a href="https://colab.research.google.com/github/luansalama/Cloud-Blender/blob/main/Improved%20Blender%20Script.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup
**Make sure to read the instructions carefully!**

If you have other resources used in the Blender project and chose to *make all paths relative*, pack all of them into a zip archive. Alternatively, you can *pack all external file*.

* `blender_version` : Version of blender used to render the scene. You may define your own blender version.
* `blend_file_path` : Path to the blend file after unpacking the zip archive. If blend file is used, this is automatically ignored.
___
* `upload_type` : Select the type of upload method. `gdrive_relative` pulls everything from the folder specified.
* `drive_path` : Path to your blend/zip file relative to the root of your Google Drive if `google_drive` is selected. Must  state the file and its extension (.zip/.blend) **unless** `gdrive_relative` is selected.
* `url_blend` : Specify the URL to the blend/zip file if `url` is selected.
___
* `animation` : Specify whether animation or still image is rendered. If **still image** is used, put the frame number in `start_frame`.
* `start_frame, end_frame` : Specify the start and end frame for animation. You may put same value such as zero for both input to set the default frame in the blend file.
___
* `download_type` : Select the type of download method. `gdrive_direct` enables the frames to be outputted directly to Google Drive (zipping will be disabled).
* `output_name` : Name of the output frames, **do NOT include .blend!** (## for frame number)
* `zip_files` : Archive multiple animation frames automatically into a zip file.
* `drive_output_path` : Path to your frames/zip file in Google Drive.
___
* `gpu_enabled, cpu_enabled` : Toggle GPU and CPU for rendering. CPU might give a slight boost in rendering time but may varies depend on the project.
* `optix_enable` : Enable OptiX which may boost performance, may be incompatible depending on the version of blender, project and GPU allocated

After you are done, go to Runtime > Run All (Ctrl + F9) and upload your files or have Google Drive authorised below. See the [GitHub repo](https://github.com/syn73/blender-colab) for more information.

In [10]:
# ====== BLENDER RENDER CONFIGURATION ======
blender_version = '4.5.2' #@param ['2.79b', '2.83.20', '2.93.18', '3.3.21', '3.6.23', '4.2.12', '4.5.1', '4.5.2'] {allow-input: true}
blend_file_path = 'path/to/file.blend' #@param {type: 'string'}

#@markdown ---
upload_type = 'google_drive' #@param ['direct', 'google_drive', 'url', 'gdrive_relative'] {allow-input: false}
drive_path = 'Only Interior.blend' #@param {type: 'string'}
url_blend = '' #@param {type: 'string'}

#@markdown ---
animation = False #@param {type: 'boolean'}
start_frame = 460 #@param {type: 'integer'}
end_frame = 460 #@param {type: 'integer'}

#@markdown ---
download_type = 'google_drive' #@param ['direct', 'google_drive', 'gdrive_direct'] {allow-input: false}
output_name = 'blender-##' #@param {type: 'string'}
zip_files = True #@param {type: 'boolean'}
drive_output_path = 'Output' #@param {type: 'string'}

#@markdown ---
gpu_enabled = True #@param {type: 'boolean'}
optix_enabled = True #@param {type: 'boolean'}
cpu_enabled = False #@param {type: 'boolean'}

#@markdown ---
blender_extra_flags = "" #@param {type: 'string'}
# Advanced: Pass additional Blender flags (e.g., "--verbose")

In [11]:
# ====== VALIDATE INPUTS ======
print("=" * 60)
print("🔍 VALIDATING CONFIGURATION")
print("=" * 60)

errors = []

# Validate frame range
if start_frame > end_frame:
    errors.append(f"❌ start_frame ({start_frame}) > end_frame ({end_frame})")

# Validate upload settings
if upload_type == 'google_drive' and not drive_path.strip():
    errors.append(f"❌ upload_type is 'google_drive' but drive_path is empty")

if upload_type == 'url' and not url_blend.strip():
    errors.append(f"❌ upload_type is 'url' but url_blend is empty")

# Validate output settings
if not output_name.strip():
    errors.append(f"❌ output_name cannot be empty")

# Show errors if any
if errors:
    print("\n❌ Configuration errors found:\n")
    for error in errors:
        print(f"  {error}")
    print("\n⚠️  Please fix the errors above and re-run this cell.\n")
    raise SystemExit("Fix configuration errors above")

print("✅ Configuration validated - all settings OK")
print("=" * 60)

🔍 VALIDATING CONFIGURATION
✅ Configuration validated - all settings OK


In [12]:
# ====== CHECK GPU & SYSTEM ======
import os
import subprocess
from pathlib import Path
import time

print("\n" + "=" * 60)
print("🔧 SYSTEM SETUP")
print("=" * 60)

os.chdir('/content')
print(f"Working directory: {Path.cwd()}")

# Check GPU availability
print("\n📡 Checking GPU...")
try:
    result = subprocess.run(
        ['nvidia-smi', '--query-gpu=gpu_name', '--format=csv,noheader'],
        capture_output=True,
        text=True,
        timeout=10
    )
    if result.returncode == 0:
        gpu_name = result.stdout.strip()
        print(f"✅ GPU available: {gpu_name}")

        # Check for unsupported GPUs
        if "Tesla K80" in gpu_name and optix_enabled:
            print("⚠️  K80 GPU detected - OptiX not supported, using CUDA")
            optix_enabled = False
    else:
        print("⚠️  GPU not available - will use CPU (slow!)")
        gpu_enabled = False
except Exception as e:
    print(f"⚠️  Could not detect GPU: {e}")
    gpu_enabled = False

# Setup tcmalloc for memory optimization
print("\n💾 Setting up memory allocator...")
os.environ["LD_PRELOAD"] = ""

try:
    subprocess.run(['apt', 'remove', '-y', 'libtcmalloc-minimal4'],
                   capture_output=True, timeout=30)
    subprocess.run(['apt', 'install', '-y', 'libtcmalloc-minimal4'],
                   capture_output=True, timeout=30)

    tcmalloc_path = "/usr/lib/x86_64-linux-gnu/libtcmalloc_minimal.so.4.5.9"
    if Path(tcmalloc_path).exists():
        os.environ["LD_PRELOAD"] = tcmalloc_path
        print(f"✅ tcmalloc enabled: {tcmalloc_path}")
    else:
        print(f"⚠️  tcmalloc not found at {tcmalloc_path}")
except Exception as e:
    print(f"⚠️  Could not setup tcmalloc: {e}")

print("=" * 60)


🔧 SYSTEM SETUP
Working directory: /content

📡 Checking GPU...
✅ GPU available: Tesla T4

💾 Setting up memory allocator...
✅ tcmalloc enabled: /usr/lib/x86_64-linux-gnu/libtcmalloc_minimal.so.4.5.9


In [13]:
# ====== MOUNT DRIVE & UPLOAD FILES ======
import shutil
from google.colab import files, drive

print("\n" + "=" * 60)
print("📥 LOADING BLEND FILE")
print("=" * 60)

uploaded_filename = ""

# Check if we need Google Drive
if (upload_type in ['google_drive', 'gdrive_relative'] or
    download_type in ['google_drive', 'gdrive_direct']):
    print("\n📁 Mounting Google Drive...")
    drive.mount('/drive')
    print("✅ Google Drive mounted at /drive")

# Load blend file based on upload type
if upload_type == 'direct':
    print("\n📤 Uploading file directly...")
    uploaded = files.upload()
    for fn in uploaded.keys():
        uploaded_filename = fn
    print(f"✅ Uploaded: {uploaded_filename}")

elif upload_type == 'url':
    print(f"\n🌐 Downloading from URL: {url_blend}")
    result = subprocess.run(['wget', '-nc', url_blend],
                          capture_output=True, timeout=300)
    if result.returncode == 0:
        uploaded_filename = os.path.basename(url_blend)
        print(f"✅ Downloaded: {uploaded_filename}")
    else:
        print(f"❌ Download failed: {result.stderr.decode()}")
        raise SystemExit("Failed to download blend file")

elif upload_type == 'google_drive':
    print(f"\n📂 Loading from Google Drive: {drive_path}")
    try:
        shutil.copy(f'/drive/MyDrive/{drive_path}', '.')
        uploaded_filename = os.path.basename(drive_path)
        print(f"✅ Loaded: {uploaded_filename}")
    except FileNotFoundError:
        print(f"❌ File not found in Google Drive: {drive_path}")
        raise SystemExit("File not found in Google Drive")

print("=" * 60)


📥 LOADING BLEND FILE

📁 Mounting Google Drive...
Drive already mounted at /drive; to attempt to forcibly remount, call drive.mount("/drive", force_remount=True).
✅ Google Drive mounted at /drive

📂 Loading from Google Drive: Only Interior.blend
✅ Loaded: Only Interior.blend


In [14]:
# ====== PREPARE DIRECTORIES ======
from pathlib import Path

print("\n" + "=" * 60)
print("🗂️  PREPARING DIRECTORIES")
print("=" * 60)

CONTENT = Path('/content')
render_dir = CONTENT / 'render'
output_dir = CONTENT / 'output'

# Clean and create directories
if render_dir.exists():
    print(f"\n🗑️  Cleaning: {render_dir}")
    shutil.rmtree(render_dir)

render_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)
print(f"✅ Created render directory: {render_dir}")
print(f"✅ Created output directory: {output_dir}")

# Handle different upload types
if upload_type == 'gdrive_relative':
    print(f"\n📋 Copying folder: {drive_path}")
    drive_path_fixed = drive_path if drive_path.endswith('/') else drive_path + '/'
    subprocess.run(['cp', '-r', f'/drive/MyDrive/{drive_path_fixed}.', str(render_dir)],
                  check=True)
    print(f"✅ Copied files to {render_dir}")

elif uploaded_filename.lower().endswith('.zip'):
    print(f"\n📦 Extracting: {uploaded_filename}")
    subprocess.run(['unzip', '-o', uploaded_filename, '-d', str(render_dir)],
                  check=True, capture_output=True)
    print(f"✅ Extracted to {render_dir}")

elif uploaded_filename.lower().endswith('.blend'):
    print(f"\n📝 Copying blend file: {uploaded_filename}")
    shutil.copy(uploaded_filename, render_dir)
    blend_file_path = uploaded_filename
    print(f"✅ Copied to {render_dir}")

elif uploaded_filename:
    ext = Path(uploaded_filename).suffix.lower()
    print(f"\n❌ Invalid file format: {ext}")
    print(f"   Valid formats: .blend, .zip")
    raise SystemExit("Invalid file extension")

print("=" * 60)


🗂️  PREPARING DIRECTORIES

🗑️  Cleaning: /content/render
✅ Created render directory: /content/render
✅ Created output directory: /content/output

📝 Copying blend file: Only Interior.blend
✅ Copied to /content/render


In [15]:
# ====== DOWNLOAD BLENDER ======
import requests
import sys

print("\n" + "=" * 60)
print("⬇️  DOWNLOADING BLENDER")
print("=" * 60)

# Blender version URLs
blender_url_dict = {
    '2.79b'   : "https://ftp.nluug.nl/pub/graphics/blender/release/Blender2.79/blender-2.79b-linux-glibc219-x86_64.tar.bz2",
    '2.83.20' : "https://ftp.nluug.nl/pub/graphics/blender/release/Blender2.83/blender-2.83.20-linux-x64.tar.xz",
    '2.93.18' : "https://ftp.nluug.nl/pub/graphics/blender/release/Blender2.93/blender-2.93.18-linux-x64.tar.xz",
    '3.3.21'  : "https://ftp.nluug.nl/pub/graphics/blender/release/Blender3.3/blender-3.3.21-linux-x64.tar.xz",
    '3.6.23'  : "https://ftp.nluug.nl/pub/graphics/blender/release/Blender3.6/blender-3.6.23-linux-x64.tar.xz",
    '4.2.12'  : "https://ftp.nluug.nl/pub/graphics/blender/release/Blender4.2/blender-4.2.12-linux-x64.tar.xz",
    '4.5.1'   : "https://ftp.nluug.nl/pub/graphics/blender/release/Blender4.5/blender-4.5.1-linux-x64.tar.xz",
    '4.5.2'   : "https://ftp.nluug.nl/pub/graphics/blender/release/Blender4.5/blender-4.5.2-linux-x64.tar.xz",
}

# Get URL for requested version
if blender_version in blender_url_dict:
    blender_url = blender_url_dict[blender_version]
    print(f"✓ Found predefined URL for version {blender_version}")
else:
    # Auto-construct URL for custom versions
    major_minor = ".".join(blender_version.split('.')[:2])
    blender_url = f"https://ftp.nluug.nl/pub/graphics/blender/release/Blender{major_minor}/blender-{blender_version}-linux-x64.tar.xz"
    print(f"✓ Constructing URL for version {blender_version}")

# Verify URL exists
print(f"\n🔗 Checking URL accessibility...")
try:
    response = requests.head(blender_url, allow_redirects=True, timeout=10)
    if response.status_code == 200:
        base_url = os.path.basename(blender_url)
        file_size = int(response.headers.get('content-length', 0))
        file_size_mb = file_size / (1024 * 1024)
        print(f"✅ URL accessible")
        print(f"   File: {base_url}")
        print(f"   Size: {file_size_mb:.1f} MB")
    else:
        print(f"❌ Error checking URL (status {response.status_code})")
        print(f"   You may need to define the URL manually above")
        raise SystemExit("Cannot access Blender download URL")
except Exception as e:
    print(f"❌ Error checking URL: {e}")
    raise SystemExit("Cannot access Blender download URL")

# Download Blender with progress
print(f"\n📥 Downloading Blender {blender_version}...")
print(f"   This may take several minutes depending on file size...")

blender_dir = CONTENT / blender_version
blender_dir.mkdir(exist_ok=True, parents=True)

def download_with_progress(url, filename, chunk_size=8192):
    """Download file with progress bar"""
    response = requests.get(url, stream=True, timeout=300)
    total_size = int(response.headers.get('content-length', 0))

    downloaded = 0
    with open(filename, 'wb') as f:
        for chunk in response.iter_content(chunk_size=chunk_size):
            if chunk:
                f.write(chunk)
                downloaded += len(chunk)

                # Print progress every 10MB
                if downloaded % (10 * 1024 * 1024) == 0 or downloaded == total_size:
                    percent = (downloaded / total_size * 100) if total_size > 0 else 0
                    mb_downloaded = downloaded / (1024 * 1024)
                    mb_total = total_size / (1024 * 1024)

                    # Progress bar
                    bar_length = 30
                    filled = int(bar_length * downloaded / total_size) if total_size > 0 else 0
                    bar = '█' * filled + '░' * (bar_length - filled)

                    print(f"\r   [{bar}] {percent:.1f}% ({mb_downloaded:.1f}/{mb_total:.1f} MB)", end='', flush=True)

    print()  # New line after progress
    return filename

# Download
download_with_progress(blender_url, base_url)
print(f"✅ Download complete: {base_url}")

# Extract Blender with progress
print(f"\n📦 Extracting Blender to {blender_dir}...")
print(f"   This may take a couple minutes...")

# Show tar progress
result = subprocess.run([
    'tar', '-xvf', base_url,
    '-C', str(blender_dir),
    '--strip-components=1'
], capture_output=True, text=True)

# Count extracted files
extracted_lines = result.stdout.count('\n')
print(f"✅ Extraction complete: {extracted_lines} files extracted")
print(f"✅ Blender ready at: {blender_dir / 'blender'}")

# Verify Blender executable exists
blender_exe = blender_dir / 'blender'
if blender_exe.exists():
    print(f"✅ Verified: Blender executable found")
    # Get file size
    exe_size = blender_exe.stat().st_size / (1024 * 1024)
    print(f"   Size: {exe_size:.1f} MB")
else:
    print(f"❌ Error: Blender executable not found!")
    raise SystemExit("Blender installation failed")

print("=" * 60)


⬇️  DOWNLOADING BLENDER
✓ Found predefined URL for version 4.5.2

🔗 Checking URL accessibility...
✅ URL accessible
   File: blender-4.5.2-linux-x64.tar.xz
   Size: 359.9 MB

📥 Downloading Blender 4.5.2...
   This may take several minutes depending on file size...
   [██████████████████████████████] 100.0% (359.9/359.9 MB)
✅ Download complete: blender-4.5.2-linux-x64.tar.xz

📦 Extracting Blender to /content/4.5.2...
   This may take a couple minutes...
✅ Extraction complete: 6509 files extracted
✅ Blender ready at: /content/4.5.2/blender
✅ Verified: Blender executable found
   Size: 156.0 MB


In [16]:
# ====== GENERATE GPU CONFIGURATION SCRIPT ======

print("\n" + "=" * 60)
print("⚙️  GENERATING GPU CONFIGURATION")
print("=" * 60)

# Create the Python script for Blender GPU setup
gpu_script = f"""import re
import bpy

scene = bpy.context.scene
scene.cycles.device = 'GPU'
prefs = bpy.context.preferences
prefs.addons['cycles'].preferences.get_devices()
cprefs = prefs.addons['cycles'].preferences

print(cprefs)

# Try to set compute device type
for compute_device_type in ('CUDA', 'OPENCL', 'NONE'):
    try:
        cprefs.compute_device_type = compute_device_type
        print(f'Device type set to: {{compute_device_type}}')
        break
    except TypeError:
        pass

# Configure individual devices
for device in cprefs.devices:
    device_name = device.name.lower()

    # Enable GPU devices (except Intel)
    if 'intel' not in device_name:
        print(f'Enabling GPU: {{device.name}}')
        device.use = {gpu_enabled}
    else:
        # Intel devices use CPU
        print(f'Setting Intel to CPU: {{device.name}}')
        device.use = {cpu_enabled}

print("GPU configuration complete")
"""

script_path = CONTENT / 'setgpu.py'
with open(script_path, 'w') as f:
    f.write(gpu_script)

print(f"✅ GPU script created: {script_path}")

# Determine which renderer to use
if optix_enabled:
    renderer = "OPTIX"
    print(f"⚡ Using renderer: OPTIX (OptiX acceleration)")
    print(f"   Note: If OptiX fails, Blender will fall back to CUDA")
else:
    renderer = "CUDA"
    print(f"⚡ Using renderer: CUDA")

print("=" * 60)


⚙️  GENERATING GPU CONFIGURATION
✅ GPU script created: /content/setgpu.py
⚡ Using renderer: OPTIX (OptiX acceleration)
   Note: If OptiX fails, Blender will fall back to CUDA


In [17]:
# ====== START RENDERING ======

print("\n" + "=" * 60)
print("🎬 STARTING RENDER")
print("=" * 60)

# Set up paths
blender_exe = CONTENT / blender_version / 'blender'
blend_file = CONTENT / 'render' / blend_file_path
output_path = CONTENT / 'output' / output_name

print(f"\n📋 RENDER CONFIGURATION:")
print(f"   ┌─ Blender Executable")
print(f"   │  {blender_exe}")
print(f"   ├─ Scene File")
print(f"   │  {blend_file}")
print(f"   ├─ Output Path")
print(f"   │  {output_path}")
print(f"   ├─ Renderer")
print(f"   │  {renderer}")
print(f"   ├─ GPU")
print(f"   │  Enabled: {gpu_enabled}, OptiX: {optix_enabled}")

if animation:
    print(f"   ├─ Type: ANIMATION")
    print(f"   │  Frames: {start_frame:04d} → {end_frame:04d}")
    print(f"   │  Total frames: {end_frame - start_frame + 1}")
else:
    print(f"   ├─ Type: SINGLE FRAME")
    print(f"   │  Frame: {start_frame}")

print(f"   └─ Download Type: {download_type}")

# Verify files exist
print(f"\n✓ Verifying files...")
if not blender_exe.exists():
    print(f"❌ Blender executable not found: {blender_exe}")
    raise SystemExit("Blender not installed")
print(f"   ✅ Blender executable: OK")

if not blend_file.exists():
    print(f"❌ Blend file not found: {blend_file}")
    raise SystemExit("Blend file not found")
print(f"   ✅ Blend file: OK ({blend_file.stat().st_size / (1024*1024):.1f} MB)")

# Build Blender command
cmd = [
    str(blender_exe),
    '-b',  # Background mode (no UI)
    str(blend_file),  # Blend file to render
    '-P', str(CONTENT / 'setgpu.py'),  # Python startup script
    '-E', 'CYCLES',  # Render engine
    '-o', str(output_path),  # Output path
    '-noaudio',  # Don't process audio
]

# Add frame/animation options
if animation:
    cmd.append('-a')  # Render all frames (animation)
    if start_frame != end_frame:  # Only add frame range if different
        cmd.extend(['-s', str(start_frame), '-e', str(end_frame)])
else:
    # Single frame render
    cmd.extend(['-f', str(start_frame)])

# Add GPU device option
cmd.extend(['--', '--cycles-device', renderer])

# Add any extra flags
if blender_extra_flags.strip():
    extra_flags = blender_extra_flags.strip().split()
    cmd.extend(extra_flags)
    print(f"   ✅ Extra flags: {blender_extra_flags}")

# Show full command for debugging
print(f"\n🔧 BLENDER COMMAND:")
print(f"   {' '.join(cmd)}\n")

# Record start time
render_start_time = time.time()

print("=" * 60)
print("⏱️  RENDERING IN PROGRESS...")
print("=" * 60)
print()

# Run Blender with live output
try:
    # Run Blender and stream output in real-time
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        universal_newlines=True,
        bufsize=1
    )

    frame_count = 0
    last_frame = None

    # Print output line by line as it comes
    for line in process.stdout:
        line = line.rstrip()

        # Look for frame completion messages
        if 'Saved:' in line or 'saved' in line.lower():
            frame_count += 1
            print(f"✅ {line}")
        elif 'Error' in line or 'error' in line.lower() or 'failed' in line.lower():
            print(f"⚠️  {line}")
        elif 'Fra:' in line:  # Blender frame indicator
            print(f"🎬 {line}")
        elif line.strip():  # Print non-empty lines
            print(f"   {line}")

    # Wait for process to complete
    return_code = process.wait()

    print()
    print("=" * 60)

    if return_code == 0:
        # Calculate elapsed time
        elapsed = time.time() - render_start_time
        hours = int(elapsed // 3600)
        minutes = int((elapsed % 3600) // 60)
        seconds = int(elapsed % 60)

        print(f"✅ RENDER COMPLETE!")
        print(f"⏱️  Time elapsed: {hours:02d}h {minutes:02d}m {seconds:02d}s")

        # Show output statistics
        output_files = list((CONTENT / 'output').glob('*'))
        if output_files:
            total_size = sum(f.stat().st_size for f in output_files if f.is_file())
            total_size_mb = total_size / (1024 * 1024)
            print(f"📊 Output Statistics:")
            print(f"   Files created: {len(output_files)}")
            print(f"   Total size: {total_size_mb:.1f} MB")
            print(f"   Average frame size: {total_size_mb / len(output_files):.1f} MB")

        print("=" * 60)
    else:
        print(f"❌ RENDER FAILED!")
        print(f"   Error code: {return_code}")
        print("=" * 60)
        raise SystemExit(f"Blender rendering failed with code {return_code}")

except KeyboardInterrupt:
    print("\n\n❌ RENDER INTERRUPTED BY USER")
    print("=" * 60)
    raise SystemExit("Rendering was cancelled")

except Exception as e:
    print(f"\n\n❌ ERROR: {e}")
    print("=" * 60)
    raise SystemExit(f"Rendering error: {e}")


🎬 STARTING RENDER

📋 RENDER CONFIGURATION:
   ┌─ Blender Executable
   │  /content/4.5.2/blender
   ├─ Scene File
   │  /content/render/Only Interior.blend
   ├─ Output Path
   │  /content/output/blender-##
   ├─ Renderer
   │  OPTIX
   ├─ GPU
   │  Enabled: True, OptiX: True
   ├─ Type: SINGLE FRAME
   │  Frame: 460
   └─ Download Type: google_drive

✓ Verifying files...
   ✅ Blender executable: OK
   ✅ Blend file: OK (540.2 MB)

🔧 BLENDER COMMAND:
   /content/4.5.2/blender -b /content/render/Only Interior.blend -P /content/setgpu.py -E CYCLES -o /content/output/blender-## -noaudio -f 460 -- --cycles-device OPTIX

⏱️  RENDERING IN PROGRESS...

   Blender 4.5.2 LTS (hash ab25eae04993 built 2025-08-20 02:11:19)
   Read blend: "/content/render/Only Interior.blend"
   Info: Read library: '/Program Files/Blender Foundation/Blender 4.5/4.5/datafiles/assets/brushes/essentials_brushes-mesh_texture.blend', '//..\..\..\..\Program Files\Blender Foundation\Blender 4.5\4.5\datafiles\assets\brushe

In [18]:
# ====== PROCESS & DOWNLOAD OUTPUT ======

print("\n" + "=" * 60)
print("📦 PROCESSING OUTPUT")
print("=" * 60)

output_dir = CONTENT / 'output'
output_files = sorted(list(output_dir.glob('*')))

print(f"\n📊 Found {len(output_files)} output file(s)")

# Helper function for downloading/copying files
def save_file(source_path, download_type, drive_output_path):
    """Save file to destination (local or Drive)"""
    if download_type == 'direct':
        files.download(str(source_path))
        print(f"   ✅ Downloaded: {source_path.name}")
    else:
        dest_path = f'/drive/MyDrive/{drive_output_path}/'
        shutil.copy(str(source_path), dest_path)
        print(f"   ✅ Saved to Drive: {source_path.name}")

# Handle output based on file count
if len(output_files) == 0:
    print("❌ No output files found!")
    raise SystemExit("Rendering produced no output")

elif len(output_files) == 1:
    # Single frame render
    print(f"\n📸 Single frame render detected")
    save_file(output_files[0], download_type, drive_output_path)

else:
    # Multiple frames (animation)
    print(f"\n🎥 Animation render detected ({len(output_files)} frames)")

    if zip_files:
        # Create zip archive
        archive_name = output_name.replace('#', '').strip('-_') + '_render'
        print(f"\n📦 Creating archive: {archive_name}.zip")

        shutil.make_archive(archive_name, 'zip', str(output_dir))
        archive_path = CONTENT / f'{archive_name}.zip'

        print(f"   ✅ Archive created: {archive_path.stat().st_size / (1024*1024):.1f} MB")

        # Download/save archive
        save_file(archive_path, download_type, drive_output_path)
    else:
        # Save files individually
        print(f"\n📤 Saving {len(output_files)} files individually")
        for output_file in output_files:
            save_file(output_file, download_type, drive_output_path)

print("\n" + "=" * 60)
print("✅ ALL DONE!")
print("=" * 60)


📦 PROCESSING OUTPUT

📊 Found 4 output file(s)

🎥 Animation render detected (4 frames)

📦 Creating archive: blender_render.zip
   ✅ Archive created: 101.1 MB
   ✅ Saved to Drive: blender_render.zip

✅ ALL DONE!


## Disclaimer
Google Colab is targeted to researchers and students to run AI/ML tasks, data analysis and education, not rendering 3D scenes. Because the computing power provided are free, the usage limits, idle timeouts and speed of the rendering may varies time by time. [Colab Pro and Colab Pro+](https://colab.research.google.com/signup) are available for those who wanted to have more powerful GPU and longer runtimes for rendering. See the [FAQ](https://research.google.com/colaboratory/faq.html) for more info. In some cases, it might be faster to use an online Blender renderfarm.

## License
```
MIT License

Copyright (c) 2020-2022 ynshung

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
```